# Training optimization: PyTorch & Optuna

## Notebook setup

### Imports

In [ ]:
# Standard library imports
import pickle

# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.optim as optim
from torchvision import datasets, transforms

# Package imports
import image_classification_tools.pytorch.data as data_utils
import image_classification_tools.pytorch.evaluation as eval_utils
import image_classification_tools.pytorch.hyperparameter_optimization as optimization
import image_classification_tools.pytorch.plotting as plots
import image_classification_tools.pytorch.training as training

# Local imports
import configuration as config

### Fixed hyperparameters

In [ ]:
# Optuna settings
run_optimization = True  # Run optimization (True) or load results for evaluation (False)
start_new_study = True   # Clear results/restart (True) or resume previous run (False)
validation_size = 10000
n_trials = 100           # Number of optimization trials
n_epochs_per_trial = 50  # Epochs per trial (reduced for faster iteration)
n_epochs_final = 200     # Epochs for final model training with optimized hyperparameters
print_every = 10         # Print training progress every n epochs

# Fixed batch size from architecture optimization
batch_size = 128

## 1. Load optimized architecture from notebook 04

We'll load the best architecture hyperparameters from the previous optimization study.

In [ ]:
# Load architecture optimization study from notebook 04
architecture_study = optuna.load_study(
    study_name='cnn_training_optimization',
    storage=config.OPTUNA_STORAGE_URL
)

# Get best architecture parameters
arch_params = architecture_study.best_trial.params

print('Best architecture hyperparameters from notebook 04:\n')
for key, value in arch_params.items():
    print(f'  {key}: {value}')

print(f'\nBest validation accuracy: {architecture_study.best_trial.value:.2f}%')

## 2. Visualize CIFAR-10 sample images

CIFAR-10 contains 32x32 color images (3 channels) across 10 classes.

In [ ]:
# Define transform (RGB)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# Get a sample dataset for visualization
sample_dataset = datasets.CIFAR10(
    root=config.DATA_DIR,
    train=True,
    transform=transform
)

# Plot first 10 images from the training dataset
fig, axes = plots.plot_sample_images(sample_dataset, config.CLASS_NAMES)
plt.show()

## 3. Prepare datasets

Create train/validation/test splits with the fixed batch size.

In [ ]:
# Load datasets
train_dataset = data_utils.load_dataset(
    data_source=datasets.CIFAR10,
    transform=transform,
    root=config.DATA_DIR,
    train=True
)

test_dataset = data_utils.load_dataset(
    data_source=datasets.CIFAR10,
    transform=transform,
    root=config.DATA_DIR,
    train=False
)

# Prepare splits
train_dataset, val_dataset, test_dataset = data_utils.prepare_splits(
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    val_size=validation_size
)

print(f'Training samples: {len(train_dataset):,}')
print(f'Validation samples: {len(val_dataset):,}')
print(f'Test samples: {len(test_dataset):,}')

## 4. Optuna training hyperparameter optimization

### 4.1. Define training hyperparameter search space

We'll search over three optimizers (Adam, SGD, RMSprop) and their relevant hyperparameters.

In [ ]:
# Define training hyperparameter search space
training_search_space = {
    'optimizer': ['Adam', 'SGD', 'RMSprop'],
    'learning_rate': (1e-5, 1e-2, 'log'),
    # SGD-specific
    'momentum': (0.0, 0.99),              # For SGD
    'nesterov': [True, False],             # For SGD with momentum
    'weight_decay': (1e-6, 1e-3, 'log'),  # L2 regularization
    # Adam-specific
    'beta1': (0.8, 0.999),                 # For Adam
    'beta2': (0.9, 0.9999),                # For Adam
    # RMSprop-specific
    'alpha': (0.9, 0.999),                 # For RMSprop
}

print('Training hyperparameter search space:\n')
for key, value in training_search_space.items():
    print(f'  {key}: {value}')

### 4.2. Create objective function for training optimization

In [ ]:
def create_training_objective(train_dataset, val_dataset, arch_params, batch_size, n_epochs, device):
    """
    Create an Optuna objective function for training hyperparameter optimization.
    
    Args:
        train_dataset: Training dataset
        val_dataset: Validation dataset
        arch_params: Fixed architecture hyperparameters from notebook 04
        batch_size: Fixed batch size
        n_epochs: Number of epochs to train per trial
        device: Device to train on (cpu/cuda)
    
    Returns:
        objective: Optuna objective function
    """
    
    def objective(trial):
        # Sample optimizer type
        optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
        
        # Sample common hyperparameters
        lr = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
        
        # Create dataloaders
        train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True
        )
        val_loader = torch.utils.data.DataLoader(
            val_dataset, batch_size=batch_size, shuffle=False
        )
        
        # Create model with fixed architecture from notebook 04
        model = optimization.create_cnn(
            n_conv_blocks=arch_params['n_conv_blocks'],
            initial_filters=arch_params['initial_filters'],
            n_fc_layers=arch_params['n_fc_layers'],
            conv_dropout_rate=arch_params['conv_dropout_rate'],
            fc_dropout_rate=arch_params['fc_dropout_rate'],
            num_classes=10,
            in_channels=3
        ).to(device)
        
        # Create optimizer based on sampled type
        if optimizer_name == 'Adam':
            beta1 = trial.suggest_float('beta1', 0.8, 0.999)
            beta2 = trial.suggest_float('beta2', 0.9, 0.9999)
            optimizer = optim.Adam(
                model.parameters(),
                lr=lr,
                betas=(beta1, beta2),
                weight_decay=weight_decay
            )
        
        elif optimizer_name == 'SGD':
            momentum = trial.suggest_float('momentum', 0.0, 0.99)
            nesterov = trial.suggest_categorical('nesterov', [True, False]) if momentum > 0 else False
            optimizer = optim.SGD(
                model.parameters(),
                lr=lr,
                momentum=momentum,
                nesterov=nesterov,
                weight_decay=weight_decay
            )
        
        elif optimizer_name == 'RMSprop':
            alpha = trial.suggest_float('alpha', 0.9, 0.999)
            momentum = trial.suggest_float('momentum', 0.0, 0.99)
            optimizer = optim.RMSprop(
                model.parameters(),
                lr=lr,
                alpha=alpha,
                momentum=momentum,
                weight_decay=weight_decay
            )
        
        # Loss function
        criterion = torch.nn.CrossEntropyLoss()
        
        # Train model
        history = training.train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            epochs=n_epochs,
            print_every=None,  # Suppress output during optimization
            enable_early_stopping=True,
            early_stopping_patience=5
        )
        
        # Return best validation accuracy
        best_val_acc = max(history['val_accuracy'])
        
        # Report intermediate values for pruning
        for epoch, val_acc in enumerate(history['val_accuracy']):
            trial.report(val_acc, epoch)
            
            # Handle pruning
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return best_val_acc
    
    return objective

In [ ]:
# Create objective function
training_objective = create_training_objective(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    arch_params=arch_params,
    batch_size=batch_size,
    n_epochs=n_epochs_per_trial,
    device=config.DEVICE
)

print('Training optimization objective function created')

### 4.3. Run training optimization

In [ ]:
%%time

if run_optimization:
    print('Running training hyperparameter optimization...')
    
    # Delete existing study if desired & it exists
    if start_new_study == True:
        print('Starting new study')
        try:
            optuna.delete_study(study_name='training_optimization', storage=config.OPTUNA_STORAGE_URL)
            print('Deleted existing study')
        except KeyError:
            print('No existing study found')
    else:
        if config.OPTUNA_DB_PATH.exists():
            print(f'Resuming study from {config.OPTUNA_DB_PATH}')
        else:
            print(f'No prior results found at {config.OPTUNA_DB_PATH}, starting new study')
    
    # Create Optuna study with SQLite storage (maximize validation accuracy)
    study = optuna.create_study(
        direction='maximize',
        study_name='training_optimization',
        storage=config.OPTUNA_STORAGE_URL,
        load_if_exists=True,  # Resume if study already exists
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
    )
    
    print(f'Study stored at: {config.OPTUNA_DB_PATH}')
    
    # Run optimization
    study.optimize(training_objective, n_trials=n_trials, show_progress_bar=True)

else:
    # Load results from disk
    study = optuna.load_study(
        study_name='training_optimization',
        storage=config.OPTUNA_STORAGE_URL
    )
    print(f'Study loaded from: {config.OPTUNA_DB_PATH}')

print(f'\nBest validation accuracy: {study.best_trial.value:.2f}%')
print('\nBest training hyperparameters:')
for key, value in study.best_trial.params.items():
    print(f'  {key}: {value}')
print()

### 4.4. Visualize optimization results

In [ ]:
fig, axes = plots.plot_optimization_results(study)
plt.show()

### 4.5. Analyze optimizer performance

In [ ]:
# Analyze performance by optimizer type
optimizer_results = {'Adam': [], 'SGD': [], 'RMSprop': []}

for trial in study.trials:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        opt_name = trial.params.get('optimizer')
        if opt_name:
            optimizer_results[opt_name].append(trial.value)

print('Optimizer performance summary:\n')
for opt_name, results in optimizer_results.items():
    if results:
        print(f'{opt_name}:')
        print(f'  Trials: {len(results)}')
        print(f'  Mean accuracy: {np.mean(results):.2f}%')
        print(f'  Std accuracy: {np.std(results):.2f}%')
        print(f'  Best accuracy: {np.max(results):.2f}%')
        print()

## 5. Train final model with best training hyperparameters

### 5.1. Load winning training hyperparameters

In [ ]:
# Extract best training hyperparameters
best_training_params = study.best_trial.params

print('Best training hyperparameters:\n')
for key, value in best_training_params.items():
    print(f'  {key}: {value}')

### 5.2. Create final dataloaders

In [ ]:
# Create dataloaders with preloading for faster training
train_loader, val_loader, test_loader = data_utils.create_dataloaders(
    train_dataset, val_dataset, test_dataset,
    batch_size=batch_size,
    preload_to_memory=True,
    device=config.DEVICE
)

print(f'Dataloaders created with batch size: {batch_size}')

### 5.3. Create final model with optimized architecture

In [ ]:
# Create model with best architecture from notebook 04
final_model = optimization.create_cnn(
    n_conv_blocks=arch_params['n_conv_blocks'],
    initial_filters=arch_params['initial_filters'],
    n_fc_layers=arch_params['n_fc_layers'],
    conv_dropout_rate=arch_params['conv_dropout_rate'],
    fc_dropout_rate=arch_params['fc_dropout_rate'],
    num_classes=len(config.CLASS_NAMES),
    in_channels=3
).to(config.DEVICE)

# Get total trainable parameters
trainable_params = sum(p.numel() for p in final_model.parameters() if p.requires_grad)

print(f'{final_model}\n')
print(f'Total parameters: {trainable_params:,}')

### 5.4. Create optimizer with best training hyperparameters

In [ ]:
# Create optimizer based on best configuration
optimizer_name = best_training_params['optimizer']
lr = best_training_params['learning_rate']
weight_decay = best_training_params['weight_decay']

if optimizer_name == 'Adam':
    beta1 = best_training_params['beta1']
    beta2 = best_training_params['beta2']
    final_optimizer = optim.Adam(
        final_model.parameters(),
        lr=lr,
        betas=(beta1, beta2),
        weight_decay=weight_decay
    )
    print(f'Using Adam optimizer: lr={lr:.6f}, beta1={beta1:.4f}, beta2={beta2:.4f}, weight_decay={weight_decay:.6f}')

elif optimizer_name == 'SGD':
    momentum = best_training_params['momentum']
    nesterov = best_training_params.get('nesterov', False)
    final_optimizer = optim.SGD(
        final_model.parameters(),
        lr=lr,
        momentum=momentum,
        nesterov=nesterov,
        weight_decay=weight_decay
    )
    print(f'Using SGD optimizer: lr={lr:.6f}, momentum={momentum:.4f}, nesterov={nesterov}, weight_decay={weight_decay:.6f}')

elif optimizer_name == 'RMSprop':
    alpha = best_training_params['alpha']
    momentum = best_training_params['momentum']
    final_optimizer = optim.RMSprop(
        final_model.parameters(),
        lr=lr,
        alpha=alpha,
        momentum=momentum,
        weight_decay=weight_decay
    )
    print(f'Using RMSprop optimizer: lr={lr:.6f}, alpha={alpha:.4f}, momentum={momentum:.4f}, weight_decay={weight_decay:.6f}')

# Set cross-entropy loss
criterion = torch.nn.CrossEntropyLoss()

### 5.5. Train final model

In [ ]:
%%time

history = training.train_model(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=final_optimizer,
    device=config.DEVICE,
    epochs=n_epochs_final,
    print_every=print_every,
    enable_early_stopping=True,
    early_stopping_patience=10
)

print()

### 5.6. Learning curves

In [ ]:
fig, axes = plots.plot_learning_curves(history)
plt.show()

## 6. Evaluate final model on test set

### 6.1. Calculate test accuracy

In [ ]:
test_accuracy, predictions, true_labels = eval_utils.evaluate_model(
    final_model,
    test_loader
)

print(f'Test accuracy: {test_accuracy:.2f}%')

### 6.2. Per-class accuracy

In [ ]:
# Calculate per-class accuracy
class_correct = {name: 0 for name in config.CLASS_NAMES}
class_total = {name: 0 for name in config.CLASS_NAMES}

for pred, true in zip(predictions, true_labels):
    class_name = config.CLASS_NAMES[true]
    class_total[class_name] += 1
    
    if pred == true:
        class_correct[class_name] += 1

print('Per-class accuracy:')
print('-' * 30)

for name in config.CLASS_NAMES:
    acc = 100 * class_correct[name] / class_total[name]
    print(f'{name:12s}: {acc:.2f}%')

### 6.3. Confusion matrix

In [ ]:
fig, ax = plots.plot_confusion_matrix(true_labels, predictions, config.CLASS_NAMES)
plt.show()

### 6.4. Predicted class probability distributions

In [ ]:
# Get predicted probabilities for all test samples
final_model.eval()
all_probs = []

with torch.no_grad():
    for images, _ in test_loader:
        outputs = final_model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)

# Plot probability distributions
fig, axes = plots.plot_class_probability_distributions(all_probs, config.CLASS_NAMES)
plt.show()

### 6.5. Evaluation curves

In [ ]:
fig, (ax1, ax2) = plots.plot_evaluation_curves(true_labels, all_probs, config.CLASS_NAMES)
plt.show()

## 7. Save optimized model

Save the model trained with optimized training hyperparameters.

In [ ]:
# Save trained model
model_path = config.MODELS_DIR / 'training_optimized_cnn.pth'
torch.save(final_model, model_path)

print(f'Model saved to: {model_path}')
print(f'Test accuracy: {test_accuracy:.2f}%')

## 8. Save test results for comparison

In [ ]:
# Save performance results
results_path = config.RESULTS_DIR / 'training_optimized_cnn_results.pkl'

# Count model parameters
total_params = sum(p.numel() for p in final_model.parameters())
trainable_params = sum(p.numel() for p in final_model.parameters() if p.requires_grad)

# Create results dictionary
results_dict = {
    'true_labels': true_labels,
    'predictions': predictions,
    'all_probs': all_probs,
    'test_accuracy': test_accuracy,
    'total_params': total_params,
    'trainable_params': trainable_params,
    'best_training_params': best_training_params
}

# Save results
with open(results_path, 'wb') as f:
    pickle.dump(results_dict, f)

print(f'Test results saved to: {results_path}')
print(f'  - Test accuracy: {test_accuracy:.2f}%')
print(f'  - Total parameters: {total_params:,}')
print(f'  - Trainable parameters: {trainable_params:,}')
print(f'  - Optimizer: {best_training_params["optimizer"]}')

## Summary

This notebook optimized training hyperparameters for the CIFAR-10 CNN model:

1. **Loaded architecture** from notebook 04's optimization study
2. **Searched optimizers**: Adam, SGD, and RMSprop with their respective hyperparameters
3. **Optimized hyperparameters**: learning rate, momentum, weight decay, beta values, alpha, etc.
4. **Trained final model** with the best training configuration
5. **Evaluated performance** on the test set
6. **Saved model and results** for comparison with other approaches